First, you need to mount your Google Drive to access files stored there. This will prompt you to authenticate your Google account.

# Completed: Steps 1–3

## Step 1: Domain Selection

Selected domain: Healthcare FAQ Assistant

Business problem: General-purpose LLMs often give generic answers for healthcare questions. This project builds a domain-specific medical assistant that can better understand healthcare terminology and answer common healthcare questions more clearly and safely.

## Step 2: Non-Instruction Dataset

Created a raw healthcare text dataset:

data/non_instruction_data.txt

The dataset contains 50+ healthcare paragraphs covering diabetes, hypertension, infection control, cancer, liver disease, cardiovascular disease, EHR, vaccines, asthma, chronic disease management, and patient safety.

## Step 3: Non-Instruction Fine-Tuning

Completed Stage 1 fine-tuning using:

- Meta-Llama-3.1-8B
- Unsloth
- QLoRA / 4-bit quantization
- LoRA adapters
- Raw healthcare text
- Google Colab GPU
- Google Drive project structure

Saved model adapter:

saved_models/non_instruction_adapter

Stage 1 testing was completed using healthcare questions. This model is not yet fully instruction-tuned, but it has been adapted to healthcare domain language.

In [ ]:
!pip install trl
!pip install unsloth
!pip install bitsandbytes

In [13]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Project root in Google Drive
project_root = "/content/drive/MyDrive/healthcare-ai-assistant"

# Project folders
folders = [
    "data",
    "notebooks",
    "reports",
    "saved_models",
    "assets"
]

# Create project structure
for folder in folders:
    os.makedirs(os.path.join(project_root, folder), exist_ok=True)

print("✅ Project structure created!")
print(f"Project Root: {project_root}")

# Frequently used paths
DATA_DIR = os.path.join(project_root, "data")
NOTEBOOK_DIR = os.path.join(project_root, "notebooks")
REPORT_DIR = os.path.join(project_root, "reports")
MODEL_DIR = os.path.join(project_root, "saved_models")
ASSET_DIR = os.path.join(project_root, "assets")

Mounted at /content/drive
✅ Project structure created!
Project Root: /content/drive/MyDrive/healthcare-ai-assistant


In [14]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import os

In [15]:
max_seq_length = 2048
load_in_4bit = True

In [16]:
file_path = os.path.join(DATA_DIR, "non_instruction_data.txt")


# High-quality raw domain text for non-instruction fine-tuning
raw_content = """Employees in healthcare settings must follow strict infection control protocols. Hand hygiene is the most important measure to prevent hospital-acquired infections.

Type 2 diabetes mellitus is a chronic condition characterized by insulin resistance and relative insulin deficiency. Patients are advised to maintain a healthy diet, exercise regularly, and monitor blood glucose levels.

Hypertension, commonly known as high blood pressure, increases the risk of heart disease and stroke. Lifestyle modifications include reducing salt intake and engaging in physical activity.

The common cold is a viral infection of the upper respiratory tract. Symptoms typically include runny nose, sore throat, cough, and mild fever. It usually resolves within a week.

Antibiotic resistance is a major public health threat caused by overuse and misuse of antibiotics. Healthcare providers should prescribe them only when bacterial infection is confirmed.

Annual flu vaccination is recommended for everyone over 6 months of age, especially high-risk groups such as the elderly and people with chronic conditions.

Patient confidentiality is protected under HIPAA regulations. Protected health information must not be disclosed without proper authorization.

In emergency medicine, the ABC approach stands for Airway, Breathing, and Circulation. Immediate assessment of these is critical in trauma cases.

Chronic Kidney Disease (CKD) has five stages based on glomerular filtration rate (eGFR). Early detection can slow progression.

Depression is a common mental health disorder that can be effectively treated with psychotherapy and antidepressant medications such as SSRIs.

A balanced diet rich in fruits, vegetables, whole grains, and lean proteins supports overall health and recovery from illness.

Asthma is a chronic inflammatory disease of the airways. Triggers include allergens, exercise, and cold air. Inhalers are the mainstay of treatment.

Vaccines have dramatically reduced the incidence of diseases like measles, polio, and tetanus.

Heart failure occurs when the heart cannot pump blood effectively. Symptoms include shortness of breath, fatigue, and swelling in the legs.

Proper wound care is essential to prevent infection. Cleaning, dressing, and monitoring for signs of infection are key steps.

Osteoporosis is a condition where bones become weak and brittle. Calcium and vitamin D intake along with weight-bearing exercise help maintain bone density.

Smoking is a leading cause of preventable death. It significantly increases risks of lung cancer, COPD, and cardiovascular disease.

Prenatal care is crucial for healthy pregnancy outcomes. Regular check-ups help monitor fetal development and maternal health.

Mental health first aid training helps individuals support others experiencing mental health crises.

Telemedicine has expanded access to healthcare, especially in rural areas.

Allergic reactions can range from mild hives to life-threatening anaphylaxis. Epinephrine auto-injectors are used for severe cases.

Cancer screening programs such as mammography and colonoscopy improve early detection rates.

Physical therapy plays a vital role in rehabilitation after surgery or injury.

Medication reconciliation is important during hospital admission and discharge to prevent adverse drug events.

Handwashing with soap and water for at least 20 seconds is one of the best ways to prevent the spread of germs.

Obesity is a complex disease that increases risk for many chronic conditions. Behavioral, pharmacological, and surgical interventions may be used.

Sleep hygiene practices promote better sleep quality and overall health.

Anticoagulant therapy requires careful monitoring to prevent bleeding complications.

Palliative care focuses on improving quality of life for patients with serious illnesses.

Infection prevention bundles are used in hospitals to reduce catheter-associated and ventilator-associated infections.

Healthcare workers should receive regular training on infection prevention and control.

Electronic Health Records (EHR) improve care coordination but require strong cybersecurity measures.

Nutritional support is critical for patients in intensive care units.

Post-operative care includes pain management, mobility exercises, and monitoring for complications.

Community health programs focus on preventive care and health education.

Immunization schedules are designed based on age and risk factors to maximize protection.

Patient education on medication adherence improves treatment outcomes.

Lifestyle medicine emphasizes the role of diet, exercise, stress management, and sleep.

Radiology plays a key role in diagnosis through X-rays, CT scans, and MRIs.

Laboratory tests provide essential information for accurate diagnosis and monitoring.

Occupational health services protect workers from workplace hazards.

Public health surveillance helps track and control disease outbreaks.

Ethical considerations in healthcare include informed consent and end-of-life decisions.

Rehabilitation services help patients regain function after stroke or injury.

Maternal and child health programs aim to reduce mortality rates.

Chronic pain management often requires a multidisciplinary approach.

Geriatric care addresses the unique needs of older adults.

Precision medicine tailors treatment to individual genetic profiles.

Health literacy is important for patients to understand and follow medical advice.

Environmental factors significantly impact public health.

Disaster preparedness is essential for hospitals and healthcare systems.

Quality improvement initiatives continuously enhance patient safety and outcomes.

Interprofessional collaboration improves patient care in complex cases.

Health equity aims to ensure fair access to healthcare for all populations."""

# Write to file
with open(file_path, "w", encoding="utf-8") as f:
    f.write(raw_content)
print("✅ Dataset created!")

with open(file_path, "r", encoding="utf-8") as f:
    print("Total lines:", len(f.readlines()))

print("Total characters:", len(raw_content))

✅ Dataset created!
Total lines: 107
Total characters: 5833


In [19]:
with open(file_path, "a", encoding="utf-8") as f:
    more_content = """

Healthcare organizations follow strict protocols to ensure patient safety and quality of care. This includes regular staff training, equipment sterilization, and continuous quality improvement initiatives.

Chronic diseases such as diabetes, hypertension, and heart disease require long-term management. Patient education, lifestyle modification, and regular medical follow-up are essential components of effective care.

The liver performs many vital functions including detoxification, protein synthesis, and production of biochemicals necessary for digestion. Common liver diseases include fatty liver disease, hepatitis, and cirrhosis.

Cancer is a group of diseases characterized by uncontrolled cell growth. Metastasis occurs when cancer cells spread from the primary tumor to other parts of the body, commonly to the lungs, liver, bones, and brain.

Cardiovascular diseases are the leading cause of death globally. As people age, the risk increases due to factors like atherosclerosis, high blood pressure, and reduced heart efficiency.

Electronic Health Records (EHR) systems have transformed healthcare delivery by improving information sharing while maintaining patient privacy and security standards.

"""
    f.write(more_content)

print("✅ Added more detailed paragraphs.")

# Verify the updated file
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

✅ Added more detailed paragraphs.
Total lines: 134


## Stage 1: Non-Instruction Fine-Tuning Notebook Code

In [ ]:
import os
from datasets import Dataset
from unsloth import FastLanguageModel
import torch
import trl
import trl.trainer.sft_config as sft_config
trl.trainer.sft_config.SFTConfig = sft_config.SFTConfig

In [20]:
from datasets import Dataset
from unsloth import FastLanguageModel

# Load raw text from Google Drive project path
with open(file_path, "r", encoding="utf-8") as f:
    raw_texts = [line.strip() for line in f.readlines() if line.strip()]

print("Loaded", len(raw_texts), "paragraphs for non-instruction tuning.")

# Create Hugging Face dataset
non_inst_dataset = Dataset.from_dict({"text": raw_texts})

# Load model
max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)


print("✅ Raw text loaded and base model ready for training.")

Loaded 66 paragraphs for non-instruction tuning.
==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


✅ Raw text loaded and base model ready for training.


#### Apply LoRA + Train (Non-Instruction)

In [21]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import os

# model is in training mode
FastLanguageModel.for_training(model)

# Apply LoRA / QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# Output folder in Google Drive
non_instruction_output_dir = os.path.join(MODEL_DIR, "non_instruction_adapter")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=non_inst_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=os.path.join(project_root, "outputs_non_instruction"),
        report_to="none",
        save_strategy="no",
    ),
)

print("🚀 Starting Stage 1: Non-Instruction Fine-Tuning...")
trainer.train()

# Save LoRA adapter + tokenizer to Google Drive
model.save_pretrained(non_instruction_output_dir)
tokenizer.save_pretrained(non_instruction_output_dir)

print("✅ Stage 1 completed and saved to:")
print(non_instruction_output_dir)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.7.1 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/66 [00:00<?, ? examples/s]

🚀 Starting Stage 1: Non-Instruction Fine-Tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66 | Num Epochs = 3 | Total steps = 27
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.218019
20,1.538158


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/healthcare-ai-assistant/saved_models/non_instruction_adapter/tokenizer_config.json.


✅ Stage 1 completed and saved to:
/content/drive/MyDrive/healthcare-ai-assistant/saved_models/non_instruction_adapter


### Inferencing / Test the Model After Stage 1
---------------------------------------------------------------


In [22]:
# ==============================
# Stage 1 Inference Test
# ==============================

# Enable faster inference
FastLanguageModel.for_inference(model)

def generate_response(question, max_new_tokens=300):
    prompt = f"""### Instruction:
You are a knowledgeable medical AI assistant. Answer clearly and accurately.

### Question:
{question}

### Response:"""

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in full_response:
        answer = full_response.split("### Response:")[-1].strip()
    else:
        answer = full_response.strip()

    return answer


# ==============================
# Test Questions
# ==============================

tests = [
    "What are the main symptoms of Type 2 diabetes?",
    "How can we prevent hospital-acquired infections?",
    "What are common cardiovascular diseases as people get older?",
    "What does cancer metastasis mean?",
    "What are common causes and symptoms of liver disease?"
]

print("🧪 Testing Stage 1 Non-Instruction Fine-Tuned Model")
print("Note: Stage 1 only learns healthcare domain language. It is not fully instruction-tuned yet.\n")

for i, question in enumerate(tests, 1):
    print(f"\n🔹 Test {i}")
    print("Question:", question)
    print("-" * 80)
    print("Response:")
    print(generate_response(question))
    print("=" * 80)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 Testing Stage 1 Non-Instruction Fine-Tuned Model
Note: Stage 1 only learns healthcare domain language. It is not fully instruction-tuned yet.


🔹 Test 1
Question: What are the main symptoms of Type 2 diabetes?
--------------------------------------------------------------------------------
Response:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Main symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow wound healing.

🔹 Test 2
Question: How can we prevent hospital-acquired infections?
--------------------------------------------------------------------------------
Response:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hospital-acquired infections are common. We can reduce risk by following protocols like hand hygiene, cleaning equipment, and isolation of infected patients. Antibiotic stewardship reduces unnecessary use.

🔹 Test 3
Question: What are common cardiovascular diseases as people get older?
--------------------------------------------------------------------------------
Response:


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Common cardiovascular diseases in older adults include coronary artery disease, heart failure, and atrial fibrillation. Regular exercise, healthy diet, and monitoring blood pressure are important preventive measures.

### Note:
It is important to consult with a healthcare professional for personalized advice.

🔹 Test 4
Question: What does cancer metastasis mean?
--------------------------------------------------------------------------------
Response:


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Cancer metastasis refers to the spread of cancer cells from the primary tumor to other parts of the body. It can occur through the bloodstream, lymphatic system, or direct invasion. Treatment aims to control tumor growth and prevent further spread.

🔹 Test 5
Question: What are common causes and symptoms of liver disease?
--------------------------------------------------------------------------------
Response:
Prognosis depends on the cause, severity, and treatment response. Mild cases often have good outcomes, while severe cases may require transplantation. Long


In [3]:
!find "/content/drive/MyDrive/Colab Notebooks" -name "*non*"

/content/drive/MyDrive/Colab Notebooks/non_instruction_finetunning.ipynb


In [1]:
!cp "/content/drive/MyDrive/Colab Notebooks/non_instruction_finetunnning.ipynb" \
"/content/drive/MyDrive/healthcare-ai-assistant/notebooks/non_instruction_finetuning.ipynb"

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/non_instruction_finetunnning.ipynb': No such file or directory
